### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [ ]:
# Replace Your_Dir with your path, e.g. /content/drive/MyDrive/Colab Notebooks/EMG_keyboard_NN
%cd Your_Dir/emg2qwerty

### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [ ]:
!pip install -r requirements.txt

### Step 4: TDS training (for report)

- Ensure the dataset is in `Your_Dir/emg2qwerty/data`.
- Run **one** of the three cells below (Run 1, 2, or 3). Logs and checkpoints go to `logs/YYYY-MM-DD/HH-MM-SS/...`.
- **Run 1 — TDS baseline:** TDS + CTC, no noise/gain augmentation (`transforms=log_spectrogram_baseline`).
- **Run 2 — TDS + new augmentations:** Same model with noise and gain scaling (default `log_spectrogram`).
- **Run 3 — TDS + CR-CTC:** TDS with CR-CTC loss (default transforms with noise and gain).

#### Run 1: TDS baseline (CTC, no noise/gain)

Checkpoints: `logs/.../checkpoints/`.

In [ ]:
# Run 1: TDS baseline (plain CTC, baseline transforms: no noise, no gain)
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc transforms=log_spectrogram_baseline

#### Run 2: TDS + new augmentations (noise, gain)

Same TDS+CTC with default transforms (noise + gain scaling).

In [ ]:
# Run 2: TDS + new augmentations (noise, gain scaling)
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_ctc

#### Run 3: TDS + CR-CTC

TDS with CR-CTC loss; default transforms (noise + gain).

In [ ]:
# Run 3: TDS + CR-CTC
!python -m emg2qwerty.train user=single_user cluster=basic trainer.accelerator=gpu trainer.devices=1 model=tds_conv_crctc

#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun